# Session 4 — One interface to rule them all

**Week 2 · Session 2 · Big ideas: the framework trade *and* observability**
**Project version: v1 — the LangChain port, vendor-neutral and traced**

Session 3 ended with the CTO asking how long it would take to run ShopWise on a second vendor. The
honest answer was *"rewrite all of it"* — every call, schema, token count and error class was
Gemini-shaped. Tonight that question becomes boring. One sentence carries the session:

> **A framework is a trade: you hand over direct control, and you get back a standard interface.**

First half: what LangChain is, what one string buys, where the abstraction leaks. Second half: once
every call goes through one interface, one library can watch **all** of them — and from tonight every
bug hunt starts in a trace, not in a `print`.

**How to read this** — run top to bottom; outputs are committed. A cell that needs a provider you do
not have prints what it would have shown instead of raising. Model replies vary between runs, and
tonight between vendors too — that is part of the point.

**Today's words:** framework · provider string · `AIMessage` · `usage_metadata` · `with_structured_output` ·
trace · run · `prompt_version`.


## 1. Setup

Two things are new. **The imports moved:** `from google import genai` is gone; everything comes out
of `langchain`, and which vendor answers is a *string*. **Importing `config` loads every vendor's
key:** `get_settings()` calls `load_dotenv()`, and the Groq / OpenAI / Anthropic clients read their
own keys straight from the environment. `schemas.py` is unchanged from session 3 — section 7 is where
that becomes the argument.


In [1]:
import json
import os
import subprocess
import sys
import time
import urllib.request
import warnings
from collections import Counter
from pathlib import Path

import langchain
import langchain_core
import langsmith
from langchain.chat_models import init_chat_model

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))    # the v1 snapshot, beside this notebook
from config import settings                  # noqa: E402  — this import is what loads labs/.env
from schemas import TriageResult             # noqa: E402  — session 3's file, unchanged

print(f"{settings.APP_NAME} v{settings.VERSION}  ·  {settings.ENVIRONMENT}")
print(f"langchain {langchain.__version__}  ·  langchain-core {langchain_core.__version__}"
      f"  ·  langsmith {langsmith.__version__}")
print(f"model_id: {settings.model_id}")
if not settings.has_key:
    print("\n!! No GEMINI_API_KEY in labs/.env — Gemini cells below will raise.")
    print("   Read their committed outputs; the other providers still work.")

# This notebook needs the labs/.venv kernel. On LangChain 0.3 — which is what a default
# Anaconda `python3` kernel usually has — `langchain.messages` does not exist and the Gemini
# client reads only GOOGLE_API_KEY, so a .env holding GEMINI_API_KEY looks keyless and the
# call fails forty cells later as a Google "default credentials were not found" error.
# Fail here instead, where the cause is still readable.
if int(langchain.__version__.split(".")[0]) < 1:
    raise RuntimeError(
        f"Wrong kernel: langchain {langchain.__version__} at {sys.executable}.\n"
        "Pick 'ShopWise · labs/.venv (Python 3.12)' — Kernel ▸ Change Kernel — and run again."
    )


ShopWise Support Assistant v1.0.0  ·  development
langchain 1.3.14  ·  langchain-core 1.6.1  ·  langsmith 0.10.17
model_id: google_genai:gemini-3.1-flash-lite


### 1.1 What this notebook can run on your machine

Five providers appear tonight; almost nobody has all five. Two are free today — Gemini and Groq.
Ollama is free too but needs a 2 GB download (section 5.5). The OpenAI and Anthropic rows were run on
the instructor's billed keys; their outputs are committed. This cell settles what you have, once.


In [2]:
def ollama_running(url: str = "http://localhost:11434/api/tags", timeout: float = 2.0) -> bool:
    """Is a local Ollama server answering? Ollama has no API key to check for.

    A short timeout on purpose: this runs before every Ollama cell, and a reader
    without Ollama should wait two seconds, not thirty.
    """
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception:
        return False


# provider string, the env var its client reads, what it costs you to run
PROVIDERS = [
    ("google_genai:gemini-3.1-flash-lite",  "GEMINI_API_KEY",    "free tier — the course default"),
    ("groq:openai/gpt-oss-20b",             "GROQ_API_KEY",      "free tier — open weights, hosted"),
    ("openai:gpt-5.4-mini",                 "OPENAI_API_KEY",    "billed — instructor's key here"),
    ("anthropic:claude-haiku-4-5-20251001", "ANTHROPIC_API_KEY", "billed — instructor's key here"),
    ("ollama:llama3.2:3b",                  None,                "local — no key, no network"),
]


def available(env_var: str | None) -> tuple[bool, str]:
    """Can we call this provider? Returns (yes/no, why not)."""
    if env_var is None:
        return ollama_running(), "no Ollama server answering on :11434 (see section 5.2)"
    return bool(os.environ.get(env_var)), f"no {env_var} in labs/.env"


AVAILABLE = {ps: available(env)[0] for ps, env, _ in PROVIDERS}

for ps, env, note in PROVIDERS:
    ok, why = available(env)
    print(f"  {'yes' if ok else ' no'}  {ps:<38} {note if ok else why}")
print(f"\n{sum(AVAILABLE.values())} of {len(PROVIDERS)} providers available here.")

  yes  google_genai:gemini-3.1-flash-lite     free tier — the course default
  yes  groq:openai/gpt-oss-20b                free tier — open weights, hosted
  yes  openai:gpt-5.4-mini                    billed — instructor's key here
  yes  anthropic:claude-haiku-4-5-20251001    billed — instructor's key here
  yes  ollama:llama3.2:3b                     local — no key, no network

5 of 5 providers available here.


## 2. The sprint plan nobody wants

> Leadership wants a second vendor by Friday. **How long?**

"Support OpenAI too" is not one ticket. It is a change to every place the program touches a model —
and then two of everything, forever:

| in v0.2 | welded to Google | "also OpenAI" costs |
|---|---|---|
| `build_client()` | `genai.Client`, `HttpOptions`, milliseconds | a second builder and retry policy |
| `triage()` | `generate_content` + `response_schema` + `.parsed` | OpenAI's own structured-output call |
| cost accounting | `prompt_token_count`, `candidates_token_count` | same numbers, different names |
| error handling | `errors.ClientError` / `ServerError` | a second exception hierarchy |

That is countable. Count it:


In [ ]:
V02_DIR = HERE.parent / "session-03" / "project"

# Names that only mean something to Google. A line containing one of these has to be
# written twice the day a second vendor arrives.
GEMINI_SHAPED = ("genai", "generate_content", "GenerateContentConfig", "response_schema",
                 "response_mime_type", ".parsed", "usage_metadata", "errors.ClientError",
                 "errors.ServerError", "HttpOptions", "HttpRetryOptions")


def vendor_lines(path: Path) -> list[tuple[int, str]]:
    """Lines of real code naming a Google-specific symbol (docstrings and comments skipped)."""
    hits, in_doc = [], False
    for n, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if line.count('"""') == 2:            # a one-line docstring
            continue
        if line.count('"""') == 1:
            in_doc = not in_doc
            continue
        if in_doc or line.lstrip().startswith("#"):
            continue
        if any(m in line for m in GEMINI_SHAPED):
            hits.append((n, line.strip()))
    return hits


total = 0
for py in sorted(V02_DIR.glob("*.py")):
    found = vendor_lines(py)
    total += len(found)
    for n, line in found:
        print(f"  {py.name:<10} {n:>4}  {line[:70]}")
print(f"\n{total} lines of v0.2 name a Google-only symbol — every one is a rewrite the day a second provider arrives.")


  triage.py    22  from google.genai import errors, types
  triage.py    57  response = client.models.generate_content(
  triage.py    60  config=types.GenerateContentConfig(
  triage.py    62  response_mime_type="application/json",
  triage.py    63  response_schema=schema or TriageResult,
  triage.py    67  return checked(response, f"triage {ticket['id']}"), response.usage_met
  triage.py    78  response = client.models.generate_content(
  triage.py    81  config=types.GenerateContentConfig(
  triage.py    86  return response.text, response.usage_metadata
  triage.py   108  except errors.ClientError as err:          # 4xx — our request was wro
  triage.py   111  except errors.ServerError as err:          # 5xx — their side, even af
  utils.py     11  from google import genai
  utils.py     12  from google.genai import types
  utils.py     21  def build_client() -> genai.Client:
  utils.py     23  return genai.Client(
  utils.py     24  http_options=types.HttpOptions(
  utils.py     2

Not a catastrophe — but they are in the client, both calls, the error handling and the cost report.
There is no seam to cut along; the vendor is a property of every line. And the schema does not port
either: session 3 (section 6.4) found Gemini rejects `additionalProperties: false`, which OpenAI
*requires*.

Two ways out: write the adapter yourself (a small framework with one user — you), or adopt somebody
else's and pay in a different currency: a dependency, a vocabulary that moves, one more layer between
you and the error message.

> **A framework is a trade: you accept an abstraction, and you get back a standard interface.**


## 3. What LangChain is, and what it isn't

<img src="figures/the-trade.png" width="860"
     alt="The framework trade set out as two facing columns: on the left what you hand over — direct
     control, another dependency, a vocabulary that gets renamed between versions — and on the right
     what you get back — one standard interface, any vendor behind one string, and an ecosystem.">

"LangChain" is three packages that ship and version separately:

| package | what it is | when this course uses it |
|---|---|---|
| **`langchain`** | the standard interface — models, messages, tools, structured output, agents | from tonight |
| **`langgraph`** | the control-flow engine — state, branches, loops, pausing for a human | session 10 onward |
| **`langsmith`** | the CCTV — every call recorded with inputs, outputs, latency, tokens, cost | section 8, tonight |

Underneath: `langchain-core` (plumbing; `langchain` re-exports it) and one thin package per vendor
(`langchain-google-genai`, `-openai`, ...) wrapping that vendor's own SDK. The SDK did not go away;
it moved one level down.

### 3.1 When *not* to reach for it

**Use the framework when the shared interface saves more than the abstraction costs.** One script,
one vendor, one call → raw SDK. Several providers, structured output, tools, tracing, an agent later
→ the framework. Sessions 2–3 were raw SDK on purpose: you now know what `with_structured_output` is
doing for you, which is the only way to debug it.


In [4]:
from langchain.messages import AIMessage

print("init_chat_model comes from :", init_chat_model.__module__)
print("AIMessage really lives in  :", AIMessage.__module__)
print("...and is re-exported by   : langchain.messages\n")

# The vendor SDK is still down there. The abstraction moved it, it did not remove it.
import langchain_google_genai
print("langchain-google-genai wraps:", langchain_google_genai.ChatGoogleGenerativeAI.__module__)
import google.genai
print("...which still imports       : google.genai", google.genai.__version__)

init_chat_model comes from : langchain.chat_models.base
AIMessage really lives in  : langchain_core.messages.ai
...and is re-exported by   : langchain.messages



langchain-google-genai wraps: langchain_google_genai.chat_models
...which still imports       : google.genai 2.17.0


### 3.2 The sixty-second history

LangChain v1 shipped in **October 2025** and deleted what the internet had written about: the zoo of
chains (`LLMChain`, `ConversationChain`, …) and several agent constructors, replaced by one
`create_agent` plus plain `model.invoke()`. Old answers still *run*, from a `langchain-classic`
package we do not install — which is how to spot them. Three habits:

1. **Pin.** `langchain>=1.3,<2` in `pyproject.toml`; never `pip install -U` mid-course.
2. **Docs over blogs.** `docs.langchain.com` → the page → Python tab → smallest example → run it.
3. **Read the imports.** `langchain_core` imports and `LLMChain` mean you are reading 2023.

The full "v1 vs the internet" card is in section 12.2.


## 4. One string, one call

```
init_chat_model("google_genai:gemini-3.1-flash-lite")
                 └─ provider ─┘ └────── model ──────┘
```

The left half is **who serves** the model, not who made it — `groq:openai/gpt-oss-20b` is OpenAI's
open weights served by Groq. LangChain splits on the *first* colon, so `ollama:llama3.2:3b` works.
`settings.model_id` builds the string from two fields, so no call site ever spells out a vendor.


In [5]:
model = init_chat_model(
    settings.model_id,
    timeout=settings.REQUEST_TIMEOUT_S,     # seconds. v0.2 had to convert to milliseconds itself
    max_retries=settings.MAX_RETRIES,       # transient failures only — see below
    max_tokens=settings.MAX_OUTPUT_TOKENS,  # the ceiling on one reply
)

print(type(model).__name__, "   <- a vendor class. The factory picked it from the string.")
print(f"  timeout      {model.timeout}")
print(f"  max_retries  {model.max_retries}")

ChatGoogleGenerativeAI    <- a vendor class. The factory picked it from the string.
  timeout      60.0
  max_retries  3


`init_chat_model` returned a vendor class, and `timeout=` / `max_retries=` landed on it as plain
attributes — the same two arguments mean the same thing on five vendors, where v0.2 needed an
`HttpOptions` in milliseconds and an `HttpRetryOptions` listing status codes. As before,
`max_retries` covers *transport* failures only. Now the call:


In [6]:
reply = model.invoke("Say hello to ShopWise in one short sentence.")

print(type(reply).__name__)
print(repr(reply.text))

AIMessage
'Hello, ShopWise!'


### 4.1 `.text`, `.content`, and the one with parentheses


In [7]:
print("reply.text     ", repr(reply.text))
print("  type          ", type(reply.text).__name__, "-> a str subclass, so it behaves like one\n")

print("reply.content  ", repr(reply.content)[:150])
print("  type          ", type(reply.content).__name__)
print("\nreply.content_blocks equals reply.content here:", reply.content == reply.content_blocks)

reply.text      'Hello, ShopWise!'
  type           TextAccessor -> a str subclass, so it behaves like one

reply.content   [{'type': 'text', 'text': 'Hello, ShopWise!', 'extras': {'signature': 'EnEKbwERTTIPTifUJ5ctRQF+m+E+l+xS8X+/n2V6OCEXwOsanxpfRcgf8G07DH/mGmsB0ZQ1+c7IUo5
  type           list

reply.content_blocks equals reply.content here: True


**`.text` is a property** — the concatenated text, what you want 95% of the time. **`.content` is
whatever the provider sent**: on this stack a *list of content blocks* (text, reasoning, tool calls,
images, citations). Code that does `reply.content.strip()` was written when `.content` was a string
and breaks on the first reply with a second block. `.content_blocks` is the normalised view. The
third form is deprecated but still works — which is why it survives in codebases:


In [8]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    value = reply.text()                      # the OLD method form — still works, and shouldn't
    for w in caught:
        print(f"{w.category.__name__}: {w.message}")

print("\nit even returns the right thing:", repr(value))
print("...which is exactly why this survives in codebases: nothing breaks, so nobody notices.")


it even returns the right thing: 'Hello, ShopWise!'
...which is exactly why this survives in codebases: nothing breaks, so nobody notices.


### 4.2 `usage_metadata` — the same three keys, everywhere

v0.2 read `prompt_token_count` and `candidates_token_count` — Google's names. Under the standard
interface there is one name for each, on every provider:


In [9]:
print("usage_metadata  ", reply.usage_metadata)
print("response_metadata", {k: reply.response_metadata.get(k) for k in ("model_name", "finish_reason")})

usage_metadata   {'input_tokens': 11, 'output_tokens': 5, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}}
response_metadata {'model_name': 'gemini-3.1-flash-lite', 'finish_reason': 'STOP'}


- **`usage_metadata` is standardised**: `input_tokens`, `output_tokens`, `total_tokens`, always.
  Build cost reports on it.
- **`response_metadata` is the vendor's own envelope**, deliberately not normalised. Read it for
  debugging, never for accounting.

### 4.3 The trap that costs an evening

The separator inside a provider id is an **underscore**. `google-genai` is the pip package name:


In [10]:
try:
    init_chat_model("google-genai:gemini-3.1-flash-lite")     # hyphen: the package name
except ValueError as err:
    print(f"{type(err).__name__}: {err}")

ValueError: Unable to infer model provider for model='google-genai:gemini-3.1-flash-lite'. Please specify 'model_provider' directly.

Supported providers: anthropic, anthropic_bedrock, azure_ai, azure_openai, baseten, bedrock, bedrock_converse, cohere, deepseek, fireworks, google_anthropic_vertex, google_genai, google_vertexai, groq, huggingface, ibm, litellm, meta, mistralai, nvidia, ollama, openai, openrouter, perplexity, together, upstage, xai

For help with specific providers, see: https://docs.langchain.com/oss/python/integrations/providers


The message complains about the *model*, so people check the model id; the mistake is in the other
half. Useful side effect: the error prints every supported provider id, and none contains a hyphen —
which is why `config.py`'s `provider_ids_use_underscores` validator catches
`SHOPWISE_PROVIDER=google-genai` at startup.

### 4.4 The abstraction stops at the exception


In [11]:
try:
    init_chat_model("google_genai:gemini-does-not-exist").invoke("hi")
except Exception as err:
    print(f"{type(err).__module__}.{type(err).__name__}")
    print(f"  {str(err)[:150]}")

langchain_google_genai.chat_models.ChatGoogleGenerativeAIError
  Error calling model 'gemini-does-not-exist' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-does-not-exist is not found 


A **vendor** class from the vendor package. Groq raises something else, OpenAI something else again.
v0.2 could branch on `ClientError` vs `ServerError`; keeping that precision under LangChain means
importing a vendor's exception class — the coupling you just paid to remove. v1 catches `Exception`
and prints the class name instead.

| what LangChain standardises | what stays the vendor's |
|---|---|
| the call: `.invoke()` | the exception classes |
| the message: `AIMessage`, `.text`, `.content_blocks` | `response_metadata`'s contents |
| the counts: `input_tokens` / `output_tokens` / `total_tokens` | the `*_token_details` sub-dicts |
| the knobs: `timeout=`, `max_retries=`, `max_tokens=` | which JSON Schema features are accepted |


## 5. Five providers, one loop

<img src="figures/one-interface.png" width="860"
     alt="One call fanning out to five providers — Gemini, Groq, OpenAI, Anthropic and Ollama — with
     Gemini and Groq marked as the two a participant can run free today and Ollama marked as the one
     that runs on your own machine with no key and no network.">

The payoff. One ticket, one schema, one loop; the only thing that changes per iteration is a string.
The ticket is deliberately ambiguous (*where is my order* **and** *I want a refund*). The loop uses
`with_structured_output(..., include_raw=True)` so token counts stay visible — section 7 explains.
A provider you lack is skipped; a provider that fails prints its exception and the loop carries on.


In [12]:
TICKET = {
    "id": "TCK-101",
    "subject": "Where is my order?",
    "body": "Ordered three weeks ago, ORD-5001, still nothing. I want a refund.",
}
SYSTEM = "Triage this ShopWise support ticket."


def as_block(t: dict) -> str:
    """Same delimiters as v0.2. The customer's words are data, never instructions."""
    return f"<ticket>\nSubject: {t['subject']}\n{t['body']}\n</ticket>"


RESULTS = {}     # provider string -> the include_raw dict, kept for the cells below

for provider_string, env_var, _note in PROVIDERS:
    ok, why = available(env_var)
    if not ok:
        print(f"{provider_string:<38}  — skipped, {why}")
        continue
    try:
        vendor = init_chat_model(provider_string,
                                 timeout=settings.REQUEST_TIMEOUT_S,
                                 max_retries=settings.MAX_RETRIES)
        started = time.perf_counter()
        out = vendor.with_structured_output(TriageResult, include_raw=True).invoke(
            [("system", SYSTEM), ("human", as_block(TICKET))])
        elapsed = time.perf_counter() - started
    except Exception as err:                      # one vendor failing is not the batch failing
        print(f"{provider_string:<38}  ! {type(err).__name__}: {str(err)[:60]}")
        continue

    RESULTS[provider_string] = out
    r, u = out["parsed"], out["raw"].usage_metadata
    print(f"{provider_string:<38} {elapsed:5.2f}s  {r.category:<13} {r.urgency:<7}"
          f" in={u['input_tokens']:>5}  out={u['output_tokens']:>4}")

# The tally, so the disagreement is a number rather than something you have to squint at.
cats = Counter(o["parsed"].category for o in RESULTS.values())
urgs = Counter(o["parsed"].urgency for o in RESULTS.values())
print(f"\n{len(RESULTS)} providers -> categories {dict(cats)}  ·  urgencies {dict(urgs)}")

google_genai:gemini-3.1-flash-lite      4.06s  refund        high    in=   45  out=  62


groq:openai/gpt-oss-20b                 1.08s  refund        high    in=  395  out= 397


openai:gpt-5.4-mini                     2.36s  refund        high    in=  354  out=  51


anthropic:claude-haiku-4-5-20251001     1.87s  order_status  high    in= 1003  out= 125


ollama:llama3.2:3b                      2.96s  refund        low     in=   65  out=  49

5 providers -> categories {'refund': 4, 'order_status': 1}  ·  urgencies {'high': 4, 'low': 1}


### 5.1 Read that output

- **The code did not change.** A string did. Session 3's sprint plan just ran in a `for` loop.
- **The tally line.** Identical input, more than one answer. `order_status` and `refund` are both
  defensible — a house rule (`docs/adr/0003`) decides, not the model. Run the cell twice and a single
  provider may change its own mind; "same label" is not a swap test, it is a coin toss (session 9).
- **Urgency is different.** Three weeks without an order plus a refund demand is not `low` under
  anyone's rule. A provider alone in that column is being worse, not opinionated — the ⭐ exercise.
- **Latency is one sample** over one network. Note the local 3B model is not the fast one.
- **The input-token column** is the most expensive thing on screen — section 5.3.

### 5.2 `usage_metadata`: standard on top, vendor-specific underneath


In [13]:
for provider_string, out in RESULTS.items():
    u = out["raw"].usage_metadata
    top = {k: u[k] for k in ("input_tokens", "output_tokens", "total_tokens")}
    print(provider_string)
    print(f"    standard   {top}")
    print(f"    in detail  {u.get('input_token_details')}")
    print(f"    out detail {u.get('output_token_details')}")

google_genai:gemini-3.1-flash-lite
    standard   {'input_tokens': 45, 'output_tokens': 62, 'total_tokens': 107}
    in detail  {'cache_read': 0}
    out detail None
groq:openai/gpt-oss-20b
    standard   {'input_tokens': 395, 'output_tokens': 397, 'total_tokens': 792}
    in detail  None
    out detail {'reasoning': 340}
openai:gpt-5.4-mini
    standard   {'input_tokens': 354, 'output_tokens': 51, 'total_tokens': 405}
    in detail  {'audio': 0, 'cache_read': 0}
    out detail {'audio': 0, 'reasoning': 0}
anthropic:claude-haiku-4-5-20251001
    standard   {'input_tokens': 1003, 'output_tokens': 125, 'total_tokens': 1128}
    in detail  {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}
    out detail None
ollama:llama3.2:3b
    standard   {'input_tokens': 65, 'output_tokens': 49, 'total_tokens': 114}
    in detail  None
    out detail None


**The top line is uniform** — same three keys, every vendor. **The sub-dictionaries are not, and were
never meant to be**: Gemini reports `cache_read`, Groq `reasoning` under output, Anthropic four cache
fields, Ollama nothing. That is the shape of a good abstraction: a stable core plus an explicit,
non-portable escape hatch. Write dashboards against the top level.

One number to notice: Groq's `gpt-oss-20b` spent most of its output tokens on `reasoning` — invisible
in the answer, fully billed, and why that model returns an empty reply if `max_tokens` is too low.

### 5.3 The same request, billed twenty times over

Same ticket, same schema, same thirty-word prompt — and the input-token counts differ by an order of
magnitude. Session 3 (section 10.1) measured the schema at zero input tokens on Gemini. The same
four-line test, across five vendors:


In [14]:
for provider_string, out in RESULTS.items():
    vendor = init_chat_model(provider_string, timeout=settings.REQUEST_TIMEOUT_S,
                             max_retries=settings.MAX_RETRIES)
    bare = vendor.invoke([("system", SYSTEM), ("human", as_block(TICKET))])   # no schema at all
    without = bare.usage_metadata["input_tokens"]
    with_schema = out["raw"].usage_metadata["input_tokens"]
    print(f"{provider_string:<38} no schema {without:>5}   +TriageResult {with_schema:>5}"
          f"   ({with_schema - without:+d})")

google_genai:gemini-3.1-flash-lite     no schema    45   +TriageResult    45   (+0)


groq:openai/gpt-oss-20b                no schema   113   +TriageResult   395   (+282)


openai:gpt-5.4-mini                    no schema    49   +TriageResult   354   (+305)


anthropic:claude-haiku-4-5-20251001    no schema    56   +TriageResult  1003   (+947)


ollama:llama3.2:3b                     no schema    65   +TriageResult    65   (+0)


Two providers charge **nothing** for the schema; three charge for it, one by nearly a thousand
tokens — *on every call, forever*. Strip the schema and all five sit in the same 45–113 range. The
entire spread is the schema. Why they differ: `with_structured_output` is built on whatever each
vendor offers, and the reply shows which:


In [15]:
for provider_string, out in RESULTS.items():
    raw = out["raw"]
    how = (f"forced tool call -> {raw.tool_calls[0]['name']}" if raw.tool_calls
           else "JSON in the message text")
    print(f"{provider_string:<38} {raw.usage_metadata['input_tokens']:>5} in tok   {how}")
    print(f"{'':<38}       answered as {raw.response_metadata.get('model_name')!r}")

google_genai:gemini-3.1-flash-lite        45 in tok   JSON in the message text
                                             answered as 'gemini-3.1-flash-lite'
groq:openai/gpt-oss-20b                  395 in tok   forced tool call -> TriageResult
                                             answered as 'openai/gpt-oss-20b'
openai:gpt-5.4-mini                      354 in tok   JSON in the message text
                                             answered as 'gpt-5.4-mini-2026-03-17'
anthropic:claude-haiku-4-5-20251001     1003 in tok   forced tool call -> TriageResult
                                             answered as 'claude-haiku-4-5-20251001'
ollama:llama3.2:3b                        65 in tok   JSON in the message text
                                             answered as 'llama3.2:3b'


- **JSON in the message text** — the vendor has a JSON/schema mode of its own.
- **A forced tool call** — it has none, so LangChain describes your schema as a *tool the model must
  call* and the answer arrives in `tool_calls`; `raw.text` is empty.

Mechanism and cost are different axes: OpenAI is in the text family and still bills ~300 tokens,
because its JSON mode sends the schema in the request. The genuinely free case is the provider that
enforces the schema **at the decoder** (Gemini, Ollama). Also note OpenAI answers with a *dated* model
id you did not ask for — it lives in `response_metadata`, and it is what a pricing table matches on.

> **An abstraction can hide a bill.** The code is identical, so nothing prompts you to check.
> When a cost claim matters, measure it on the provider you use, on the day you use it.

### 5.4 Setting up Groq — free, and the fast one

1. Sign in at **<https://console.groq.com>** (Google or GitHub account).
2. **API Keys → Create API Key.** Copy it; it is shown once.
3. Add `GROQ_API_KEY=gsk_...` to `labs/.env`.
4. **Restart the kernel** — `.env` is read at import time. This is the most common "but I pasted
   it" support question in the course.

The free tier is rate-limited per minute, so a loop over twenty tickets can hit a 429.

### 5.5 Setting up Ollama — no key, no network, no bill

1. Install from **<https://ollama.com/download>** (or `brew install ollama`).
2. `ollama pull llama3.2:3b` — about 2 GB, once.
3. `ollama serve` if the desktop app is not already running it.
4. `curl http://localhost:11434/api/tags` should answer — that is what `ollama_running()` checks.

Then `ollama:llama3.2:3b` works like any other provider string. A 3B model does not fail loudly: the
schema validates, the answer is just worse. The ⭐ exercise makes you find exactly where.


## 6. Messages, three ways

You have been passing a bare string to `.invoke()`. It accepts four shapes; the difference is
authoring convenience, not what reaches the model.


In [16]:
from langchain.messages import HumanMessage, SystemMessage

SHORT = "Answer in one word."

a = model.invoke("Are you ready?")                                    # 1. a plain string
b = model.invoke([{"role": "system", "content": SHORT},               # 2. OpenAI-style dicts
                  {"role": "user", "content": "Are you ready?"}])
c = model.invoke([SystemMessage(SHORT),                               # 3. typed classes
                  HumanMessage("Are you ready?")])
d = model.invoke([("system", SHORT), ("human", "Are you ready?")])    # 4. (role, text) tuples

for name, out in (("string", a), ("dicts", b), ("typed", c), ("tuples", d)):
    print(f"  {name:<7} -> {type(out).__name__:<9} {out.text!r}")

  string  -> AIMessage 'I am ready! What can I help you with today?'
  dicts   -> AIMessage 'Yes.'
  typed   -> AIMessage 'Yes.'
  tuples  -> AIMessage 'Yes.'


All four return an `AIMessage`. The first answer is longer only because a bare string carries **no
system prompt**.

| shape | use it when |
|---|---|
| a plain string | one-off, no system prompt |
| `{"role": ..., "content": ...}` dicts | the history came from JSON, a database, an OpenAI-shaped API |
| `SystemMessage` / `HumanMessage` | writing messages in Python with editor help |
| `("system", "...")` tuples | the same, shorter — what `project/triage.py` uses |

The roles are not decoration: `system` is your instructions, `human` is the customer's words, and
keeping them apart is how a model tells them apart (session 7's injection work lives on that line).

### 6.1 The import that will bite you

Old tutorials say `from langchain_core.messages import SystemMessage`. Watch what that does:


In [17]:
import langchain_core.messages as legacy

print("langchain.messages.SystemMessage is langchain_core.messages.SystemMessage:",
      SystemMessage is legacy.SystemMessage)
print("SystemMessage.__module__ :", SystemMessage.__module__)

langchain.messages.SystemMessage is langchain_core.messages.SystemMessage: True
SystemMessage.__module__ : langchain_core.messages.system


**Same class, and the old import still works** — which is worse than failing, because nothing
teaches you. `langchain_core` is plumbing; `langchain` is the public surface. Write
`from langchain.messages import …`, and fix the import first when you copy a snippet.

### 6.2 The model is still stateless


In [18]:
first = model.invoke([HumanMessage("My name is Ayesha and my order is ORD-5001.")])
print("1st reply:", first.text[:90])

second = model.invoke([HumanMessage("What is my order number?")])
print("2nd reply:", second.text[:90])
print("\n-> the second call is a different request to a stateless server. Nothing was remembered.\n")

# Memory is something YOU send. That is all it ever is.
with_history = model.invoke([
    HumanMessage("My name is Ayesha and my order is ORD-5001."),
    first,                                   # the model's own previous turn, sent back verbatim
    HumanMessage("What is my order number?"),
])
print("with history:", with_history.text[:90])

1st reply: Hello Ayesha! Thank you for providing those details.

How can I help you with order **ORD-


2nd reply: I do not have access to your personal information, purchase history, or account details. I

-> the second call is a different request to a stateless server. Nothing was remembered.



with history: Your order number is **ORD-5001**.


**Memory is a list you resend, and every resend is billed as input tokens.** That is why session 6 is
about trimming and summarising, not a `memory=` object. The framework changed the shape of the call,
not what a language model is.


## 7. Structured output, one API

Session 3 spent an evening on `TriageResult`, then found Gemini rejects the `additionalProperties`
OpenAI requires. Here is what the port cost:


In [19]:
import hashlib

s03 = HERE.parent / "session-03" / "project" / "schemas.py"
s04 = HERE / "project" / "schemas.py"


def digest(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()[:16]


print(f"session-03/project/schemas.py  {digest(s03)}")
print(f"session-04/project/schemas.py  {digest(s04)}")
print(f"\nidentical: {digest(s03) == digest(s04)}")

session-03/project/schemas.py  09ec0ded739cb6a5
session-04/project/schemas.py  09ec0ded739cb6a5

identical: True


**Byte for byte, the same file** — and it just ran against five vendors in section 5, including
two with no structured-output API at all. Model strings are easy to abstract; **schemas are where
vendors genuinely differ**, and this is the trade paying out. `prompts.yml` is unchanged too.


In [20]:
triage = model.with_structured_output(TriageResult)
result = triage.invoke([("system", SYSTEM), ("human", as_block(TICKET))])

print(type(result).__name__)
print(f"  {result.category} | {result.urgency} | {result.sentiment} | "
      f"needs_human={result.needs_human}")
print(f"  {result.summary}")

TriageResult
  refund | high | frustrated | needs_human=True
  Customer requesting refund for delayed order ORD-5001.


One method, one argument, a validated object back. Compare v0.2: a config object, a read of
`.parsed`, and a hand-written `checked()` guard. **That guard is absorbed**: `with_structured_output`
raises instead of handing back `None`.

### 7.1 The catch: your token counts just vanished

The parsed object **replaces** the message — and everything the message carried goes with it:


In [21]:
print("type          :", type(result).__name__)
print("has .text?    :", hasattr(result, "text"))
print("has usage?    :", hasattr(result, "usage_metadata"))

try:
    result.usage_metadata
except AttributeError as err:
    print(f"\nAttributeError: {err}")
    print("  -> your cost report, your latency dashboard and your token budget just went dark.")

type          : TriageResult
has .text?    : False
has usage?    : False

AttributeError: 'TriageResult' object has no attribute 'usage_metadata'
  -> your cost report, your latency dashboard and your token budget just went dark.


This catches everyone once, at the worst moment: the pipeline works and the cost report reports
nothing. `include_raw=True` is the fix, and it changes the return type:


In [22]:
out = model.with_structured_output(TriageResult, include_raw=True).invoke(
    [("system", SYSTEM), ("human", as_block(TICKET))])

print(type(out).__name__, sorted(out), "\n")
print("parsed        :", type(out["parsed"]).__name__, "->", out["parsed"].category)
print("raw           :", type(out["raw"]).__name__, "->", out["raw"].usage_metadata)
print("parsing_error :", out["parsing_error"])

dict ['parsed', 'parsing_error', 'raw'] 

parsed        : TriageResult -> order_status
raw           : AIMessage -> {'input_tokens': 45, 'output_tokens': 66, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}}
parsing_error : None


| key | holds | `None` when |
|---|---|---|
| `parsed` | your validated Pydantic object | parsing failed |
| `raw` | the `AIMessage` — `usage_metadata`, `response_metadata`, `tool_calls` | never |
| `parsing_error` | the exception, if the answer would not fit the schema | it parsed fine |

`include_raw=True` also **stops raising**: a schema violation becomes a value you can count and log
instead of an exception that ends the batch. Use the short form in a request handler where something
upstream retries; use `include_raw=True` for batches, evaluation and anything that meters cost.

### 7.2 The project, running on all of it

`project/` is rebuilt on the above: `build_model()` (in `utils.py`) is `init_chat_model` with two
keyword arguments, `triage()` is `with_structured_output`, `draft_reply()` reads `.text`, and
`config.py` gained a `PROVIDER` field. Section 11 runs it. What this section needs is one number —
how much Google is left:


In [ ]:
# Section 2's scan, on v1:
v1_total = sum(len(vendor_lines(py)) for py in sorted((HERE / "project").glob("*.py")))
print(f"v0.2  {total:>3} vendor-coupled lines of code")
print(f"v1    {v1_total:>3}")
for py in sorted((HERE / "project").glob("*.py")):
    for n, line in vendor_lines(py):
        print(f"        {py.name:<10} {n:>4}  {line[:70]}")


v0.2   19 vendor-coupled lines of code
v1      4
        config.py    65  PROVIDER: str = "google_genai"
        config.py   127  raise ValueError("must not be empty — e.g. google_genai, groq, ollama"
        config.py   138  and self.PROVIDER == "google_genai"
        triage.py   110  if settings.PROVIDER == "google_genai" and not settings.has_key:


The vendor left the program and became a string in a settings file. The one line that remains is a
startup courtesy in `main()` — `if settings.PROVIDER == "google_genai" and not settings.has_key:` —
nowhere near the call path. That is residual coupling somebody *chose*, as opposed to v0.2's, which
nobody chose.

`triage.py`'s last line says *"Tokens, latency and cost: https://smith.langchain.com"* and nothing
has explained why. **That is next.**


## 8. Two environment variables, zero code changes

Every model call in this notebook has been recorded — inputs, outputs, latency, tokens, cost —
and **nothing in `project/` was edited to make that happen.** Suspicion is the right response, so the
next cell checks it.

### 8.1 The two variables

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_PROJECT=shopwise      # optional — names the folder traces land in
```

**Sign-up (two minutes):** <https://smith.langchain.com> with a Google or GitHub account → Settings →
API Keys → Create. Put the three lines in `labs/.env` (gitignored) and **restart the kernel**.
The free tier's allowance changes often; read it on the pricing page.

Leave `LANGSMITH_TRACING=false` until the key is in the file: on without a key it warns once and then
posts traces nobody receives. These three keep their plain names (no `SHOPWISE_` prefix) because the
tracer reads them from `os.environ` itself; `config.py` mirrors them so `python config.py` can say
whether runs are being recorded.


In [24]:
# "Zero code changes" is a claim. Check it.
#
# These are the symbols you would expect to find in a program that had been instrumented
# by hand: an import of the tracing SDK, a decorator, a tracer object, a callbacks= argument.
INSTRUMENTATION = ("import langsmith", "from langsmith", "@traceable", "LangChainTracer",
                   "tracing_v2_enabled", "callbacks=", "RunTree", "collect_runs", "run_tree")

for py in sorted((HERE / "project").glob("*.py")):
    hits = [n for n, line in enumerate(py.read_text(encoding="utf-8").splitlines(), 1)
            if any(sym in line for sym in INSTRUMENTATION)]
    print(f"  v1   {py.name:<12} {len(hits)} line(s) that instrument tracing")

# The three lines that actually did it. The key is never printed — this is a notebook you commit.
print()
for key in ("LANGSMITH_TRACING", "LANGSMITH_PROJECT", "LANGSMITH_API_KEY"):
    value = os.environ.get(key, "")
    shown = f"<set, {len(value)} chars>" if key.endswith("API_KEY") and value else (value or "(unset)")
    print(f"  labs/.env    {key:<18} {shown}")

# config.py mirrors the two that matter, so the program can say what it is doing.
TRACED = settings.LANGSMITH_TRACING and bool(settings.LANGSMITH_API_KEY.get_secret_value())
print(f"\n  recording: {TRACED}   project: {settings.LANGSMITH_PROJECT!r}")

  v1   config.py    0 line(s) that instrument tracing
  v1   schemas.py   0 line(s) that instrument tracing
  v1   triage.py    0 line(s) that instrument tracing

  labs/.env    LANGSMITH_TRACING  true
  labs/.env    LANGSMITH_PROJECT  shopwise-cce-ind-tra-2026
  labs/.env    LANGSMITH_API_KEY  <set, 51 chars>

  recording: True   project: 'shopwise-cce-ind-tra-2026'


Zero instrumentation lines. The recording comes **from the standard interface**: every LangChain
chat model runs through a callback system, and `LANGSMITH_TRACING` attaches one more listener at
import time. That is the second payout of the trade — once every call goes through one interface, a
library that has never heard of your program can watch all of them.

### 8.2 What a trace is

| word | what it is |
|---|---|
| **run** | one recorded operation — a model call, a tool call, a function |
| **trace** | the tree of runs one top-level request produced |
| **project** | the folder traces land in (`LANGSMITH_PROJECT`) |

<img src="figures/anatomy-of-a-trace.png" width="860"
     alt="One recorded run opened up: a trace card listing inputs, outputs, latency, tokens, cost
     and metadata, with a leader line from each row out to a note saying what that part of the
     record is good for.">

Five of those six rows arrive free tonight; `metadata` is the one you fill in (section 9). Here is
the real thing, one row per trace:

<img src="figures/trace-list.png" width="920"
     alt="The LangSmith run list for the shopwise project: one row per trace with Name, Input,
     Output, Start Time, Latency, Tokens and Cost columns. Most rows are named RunnableSequence and
     show a ticket subject in the Input column and the TriageResult JSON in the Output column;
     several rows show no cost at all.">

Blank `Cost` rows are the local `llama3.2:3b` — no per-token bill (section 8.4). Rows named
`RunnableSequence` are `with_structured_output` showing its shape (section 8.3).


### 8.3 Open one and read it

Everything this notebook has run is in there already. Go to <https://smith.langchain.com>, open the
project named in `LANGSMITH_PROJECT`, sort by Start Time and click the newest row — that is section
7's structured triage call, recorded without a line of code asking for it.

Four things the run holds that a `print` does not: **the prompt actually sent** (delimiters and all,
not the one you think you wrote); **wall-clock latency**, retries included, the number the customer
feels; **cost already split** into input and output; and **a URL** you can paste into a bug report.

<img src="figures/trace-detail.png" width="920"
     alt="One trace opened in the LangSmith UI. The left pane shows the run tree: RunnableSequence
     containing a ChatGoogleGenerativeAI call followed by a PydanticOutputParser step at 0.00 seconds.
     The right pane shows Tools listing TriageResult, the Input with the system prompt and ticket,
     and the Output TriageResult JSON.">

```
RunnableSequence           2.41s   275 tokens / $0.0002
  ├─ ChatGoogleGenerativeAI   2.41s   275        gemini-3.1-flash-lite
  └─ PydanticOutputParser     0.00s
```

**`with_structured_output` is a small chain, not one call**: a model call plus a local parse that
took no time and cost nothing. Two things to read correctly: the **Tools** panel lists the schema
that was *bound*, whichever way the provider implements it — it is not evidence of a tool call (read
`.tool_calls` for that). And the `llm` run holds what the model said; the `RunnableSequence` above it
holds what your code produced *after* `tidy_summary` and `money_needs_a_human` ran. When they differ,
the difference is exactly the work your code is doing.

### 8.4 It prices every call — until it doesn't

v0.2 had `cost()` and `load_prices()`; v1 has neither, and the `Cost` column is why. That is a
**move**, not a saving — and here is why it is the right move. `prices.yml` carries a hand-checked
rate; the platform priced the same calls independently, and the two agree **to the last decimal**:
$0.25 in / $1.50 out per 1M tokens, computed by two parties who never spoke.

What replaces the file is better than the file: one price map shared by every service instead of one
somebody re-checks, a per-run figure you can group and chart instead of a total at the end of one
script, and **every** vendor priced where `cost()` only ever knew Gemini's. Group the run list by
model and the evening reads like this:

| `ls_provider` | `ls_model_name` | calls | tokens | cost $ |
|---|---|---:|---:|---:|
| anthropic | claude-haiku-4-5-20251001 | 1 | 405 | 0.00180100 |
| google_genai | gemini-3.1-flash-lite | 2 | 608 | 0.00079950 |
| groq | openai/gpt-oss-20b | 1 | 519 | 0.00013028 |
| ollama | llama3.2:3b | 1 | 350 | **not priced** |
| openai | gpt-5.4-mini | 1 | 166 | 0.00056325 |

Four of five priced themselves, including Groq's open-weights model. `ollama` is blank, and **that is
correct**: a model on your laptop has no per-token bill. Do not paper over it with a price of zero.

`ls_model_name` is the seam every cost decision hangs off. A fine-tuned or self-hosted model silently
loses its costing there — token counts arrive, cost does not. Fix it in **Settings → model pricing
map**: a regex matched against `ls_model_name` (use `^gpt-5\.4-mini`, not a literal — vendors append
dates). **And there is no backfill**: a run is priced when ingested and never revisited, so add the
price *before* you run the thing you want costed.

`prices.yml` stays in the repo anyway. It is how you sanity-check the platform, it is what session
9's offline scoring estimates a run with, and it carries a `verified` date the platform never will.

### 8.5 The habit

Everything so far was a call that worked. Filter the run list by **Error** and you get the others —
twelve in an hour, on the evening this notebook was run:

```
03:42:30  gemini-does-not-exist   ChatGoogleGenerativeAIError("Error calling model 'gemini-does-not-exist' (NOT_…
03:41:00  openai/gpt-oss-20b      BadRequestError('Error code: 400 … "Tool call validation failed…
```

**Nobody instrumented either of those.** The first is section 4.4's deliberately broken model id,
forty cells ago, message and timestamp intact. The second is a real intermittent flake in Groq's
forced-tool-call path — the class of bug you cannot reproduce on demand, sitting there with its full
error text.

| when something goes wrong | before tonight | from tonight |
|---|---|---|
| "why did it answer that?" | add a `print`, re-run, hope | open the run — the prompt is in it |
| "why is it slow?" | guess | sort by latency |
| "what is this costing?" | maintain a price table | read the cost column |
| "did the prompt change break it?" | argue | compare two runs — **once section 9 stamps them** |

> **From here, every bug hunt starts in the trace.**


## 9. Which prompt said that?

Session 3 changed one paragraph in `prompts.yml` and the score moved from 15/20 to 18/20. A file with
that property is production behaviour, and editing it is a **deployment** — so it inherits the
question every deployment has: *which version produced this result?*

<img src="figures/prompt-version-loop.png" width="860"
     alt="A five-station cycle showing that a prompt change is a deployment: edit the prompt in
     prompts.yml, stamp it with a version and a content hash, score it over the twenty hand-labelled
     tickets, compare the two runs in the trace, then ship or revert — and round again.">

### 9.1 Three levels, cheapest first

| | what you do | answers | costs |
|---|---|---|---|
| **1. git alone** | `git log -p prompts.yml` | *what wordings has this project had?* | nothing |
| **2. git + a stamp** | two metadata keys on every run | *this bad answer from Tuesday — which wording?* | one `config=` argument |
| **3. a hosted registry** | prompts live in a service | *who may change production wording without a PR?* | a second source of truth |

Git alone fails at exactly one question — the one you ask during an incident: it lists every prompt
that existed on Tuesday, but only a stamp on the run says which one *that run* used. ShopWise ships
level 2; the rule is that **`prompts.yml` in git stays the single source of truth** (`docs/adr/0005`).
Level 3 arrives in session 15, when somebody who cannot open a pull request needs to change wording.

### 9.2 The stamp

`prompts.yml` gained a first key, `prompt_version: v1`, and every call now carries that label plus a
hash of the prompt **as assembled**:


In [31]:
import triage as project        # the v1 snapshot's own module — not a copy of it

shipped = project.build_triage_prompt()      # what the model is actually instructed with

print(f"prompts.yml first key   prompt_version: {project.PROMPTS['prompt_version']}")
print(f"settings override       {settings.PROMPT_VERSION!r}   (empty = the file wins)")
print(f"\nso every run carries    {project.stamp(shipped)}")

# The hash is of the ASSEMBLED prompt, not of the file. Switch on the few-shot block and the
# model is instructed differently — so the hash moves, even though prompts.yml did not.
few = project.build_triage_prompt(with_examples=True)
print(f"\nsha, few-shot OFF (shipped)  {project.prompt_sha(shipped)}")
print(f"sha, few-shot ON             {project.prompt_sha(few)}")

# And the failure the hash exists to catch: an edit somebody shipped at 11pm without bumping
# the version, because that is what the file already said.
sneaky = shipped.replace("it is a complaint", "it is ALWAYS a complaint")
print(f"sha, one word edited         {project.prompt_sha(sneaky)}   <- same version, new hash")

prompts.yml first key   prompt_version: v1
settings override       ''   (empty = the file wins)

so every run carries    {'metadata': {'prompt_version': 'v1', 'prompt_sha': '41dcfc647975'}, 'tags': ['prompt:v1']}

sha, few-shot OFF (shipped)  41dcfc647975
sha, few-shot ON             1c24b721318f
sha, one word edited         7f4772745f56   <- same version, new hash


**Why both?** The *name* is what a human filters on and can say in a stand-up ("the v2 prompt
regressed refunds"). The *hash* is what catches you when the version was not bumped: nobody types
it, so it cannot be forgotten. **Same `prompt_version`, different `prompt_sha` = the wording moved
and nobody announced it.** It hashes the assembled prompt (few-shot block included when on), never
the ticket — that is data, not wording. And the file wins over `settings.PROMPT_VERSION`, which is
only an override for labelling an experiment arm.

### 9.3 The stamp, in the trace

`project.triage()` gained no new arguments — the stamp rides along on the ordinary call. Open any run
it made and the **Metadata** panel carries:

```
prompt_version   v1
prompt_sha       41dcfc647975
ls_model_name    gemini-3.1-flash-lite
tags             ['seq:step:1', 'prompt:v1']
```


`prompt:v1` in the tags is what you **click** in the run list; the metadata is what you **filter** by
in a query. One trap: the stamp lands on every run in the chain — sequence, model call, parser — so
**scope to `run_type="llm"`** when counting calls or summing tokens, or everything triples. The A/B
below makes twenty calls per arm and leaves sixty runs per arm behind it.

### 9.4 The A/B

We have a candidate wording and twenty hand-labelled tickets, so we do not have to argue. The
candidate `v2-ordered` states the **same four conventions, in the same words**, as a numbered
"first match wins" procedure instead of four prose bullets. Nothing else differs — change one thing,
or you cannot say what moved the number. It lives in `score_triage.py`, not `prompts.yml`: the
application ships one wording, the harness holds the candidates.

The harness also reads its own token counts and cost back out of the trace, through the LangSmith
SDK — the same records the platform prices production by, rather than a second price table of its
own. That read-back belongs in a tool like `score_triage.py`; nothing in production reads its own
traces back. Forty live calls:


In [5]:
# 20 tickets x 2 arms = 40 live calls. It takes a couple of minutes; that is the price of
# knowing. The harness reads its own tokens and cost out of the trace afterwards — the same
# records the platform prices production by, rather than a second price table of its own.
done = subprocess.run([sys.executable, "score_triage.py", "--prompt-version", "v2-ordered"],
                      cwd=HERE, capture_output=True, text=True, timeout=1800)
AB = done.stdout or done.stderr[-2000:]
print(AB)


python(79383) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


py311/lib/python3.11/site-packages/langchain_core/load/serializable.py", line 130, in __init__
    super().__init__(*args, **kwargs)
  File "/opt/anaconda3/envs/py311/lib/python3.11/site-packages/pydantic/main.py", line 263, in __init__
    validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/py311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py", line 1091, in validate_environment
    self.client = genaix.build_generative_service(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/py311/lib/python3.11/site-packages/langchain_google_genai/_genai_extension.py", line 276, in build_generative_service
    return v1betaGenerativeServiceClient(**config)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/py311/lib/python3.11/site-packages/google/ai/generativelanguage_v

### 9.5 Reading the result honestly

**The candidate lost**, three runs in a row:

| arm | accuracy | cost | tickets wrong |
|---|---|---|---|
| shipped `v1` — four prose bullets | **18/20** | $0.0028 | TCK-003, TCK-005 |
| `v2-ordered` — same four, numbered, first match wins | **16/20** | $0.0030 | TCK-002, TCK-003, TCK-005, TCK-006 |

So the shipped wording stays and the loser stays in the harness as evidence. Read the misses, not the
total: **"first match wins" made the complaint rule greedy** because it is first — TCK-002 (a faulty
earcup, a refund request) now stops at rule 1. An ordered procedure does not restate conventions; it
changes their precedence.

And the boring number matters most: the shipped arm scored the same 18/20 session 3 measured on the
raw SDK. The port, the tracing and the stamp cost **no measurable accuracy** — with the usual caveat
that `TCK-015` flips between runs, so the honest claim is "the same band", not "identical". One ticket
is five points on a set of twenty; session 9 builds an eval that can tell a regression from a coin flip.


### 9.6 Rollback, and the trap

Shipping a prompt is a commit; unshipping one is `git revert` and a redeploy. Wording, version and
measurement travelled together, so they come back together — and the stamp lets you *prove* the
revert took, from the run list.

The trap: a prompt edited in a hosted UI and never mirrored back to git means production runs text
that lives in no repository, and the next developer ships a fix to a file nobody deployed. Every
prompt-management scheme can fail this way. Name one source of truth and be boring about it.


## 10. The ledger

Nine sections of *what it does*. Now the bill, in four columns:

| column | what tonight showed |
|---|---|
| **shrank** | `build_client()` collapsed into `timeout=` / `max_retries=`; `checked()` absorbed because `with_structured_output` raises instead of returning `None`; tuple returns and the millisecond conversion gone |
| **moved** | `cost()` and `load_prices()` — to a platform that agreed with `prices.yml` to the last decimal, prices every vendor, and that the eval now reads too |
| **got vaguer** | one `except Exception` where there were three typed branches (the exception classes are still the vendor's); `usage_metadata` sub-dicts that are not portable; a schema bill of +0 to +947 tokens your code cannot see |
| **bought** | on Gemini, a nicer interface — `method="json_schema"` is the same decoder-side mechanism session 3 set by hand. On the four other providers, the *implementations*: per-vendor structured output you never wrote |
| **cost in accuracy** | none measurable — 17–18/20, the same band as the raw SDK |
| **cost in lines** | the prompt stamp is a handful of lines *added*; observability is cheap, not free |

> **Use a framework when the shared interface saves more than the abstraction costs.** For ShopWise
> it does, and not narrowly: five vendors behind one string, five structured-output mechanisms you
> would have learned one incident at a time, and tracing for two environment variables. For a
> fifty-line script that calls one model once, it does not — and sessions 2 and 3 were that script,
> on purpose.

#### 💼 The framework debate — split the room

| **Team A — raw SDK** | **Team B — take the framework** |
|---|---|
| The vendor's docs describe the vendor's API; with LangChain you debug through both | Five vendors, one interface; the four you have not integrated are a string away |
| A dependency that renamed its imports at v1 will do it again | A dependency you would otherwise have written yourself, with one user and no tests |
| You gave up 4xx-vs-5xx to avoid one import | Observability for two environment variables and zero code |
| The schema bill is invisible *because* of the abstraction | The tool-call fallback is a quirk you were never going to discover in time |

The tiebreak is situational: **how many models will this program ever call, and who is on call at
2am?** One model, one developer → the SDK. Several vendors, several people, a queue that must keep
answering → the interface.

#### Three interview questions

1. **What does `init_chat_model` buy you over three SDKs?** One construction call, one invoke
   signature, one message type, one `usage_metadata` top level, and per-vendor structured output you
   did not implement. Portability, not shorter code.
2. **Your agent is wrong in production and you have no traces — what did you fail to do, for two
   environment variables?** Nothing was recorded: no prompt, no tokens, no way to tell a model
   regression from a prompt edit.
3. **How do you protect a codebase against a framework's renames?** Pin it, read release notes at
   upgrade time, keep a migration card, and keep the vendor behind your own configuration
   (`settings.model_id`) so neither rename is spelled out at a call site.


## 11. Project v1

<img src="figures/v1-pipeline.png" width="860"
     alt="Version 1's pipeline, the same shape as v0.2 with the parts swapped: a ticket goes into
     one interface that any of five providers sits behind, through with_structured_output into the
     same TriageResult schema, on to a shorter draft_reply, while LangSmith watches every call
     from below.">

**The shape did not change; every box in it did.** Open the files in your editor — the notebook is
where you learn the idea, the editor is where the software lives.

| file | what it holds | why it is separate |
|---|---|---|
| `config.py` | `Settings` — provider, model, timeouts, prompt-version override, the LangSmith variables; `model_id` joins provider and model | the other boundary where unvalidated data enters, validated like the model's answer |
| `schemas.py` | `TriageResult` | the contract, and it belongs to no vendor — byte-identical to session 3's |
| `prompts.yml` | the wording, and `prompt_version` as its first key | prompt text never lives in Python; the version moves in the same commit as the wording |
| `utils.py` | `build_model()`, `load_tickets()`, `as_ticket_block()`, `prompt_sha()` | plumbing every script needs, kept out of the program |
| `triage.py` | the pipeline: assemble the prompt, stamp it, call twice | the only file that knows the *order* things happen in |
| `score_triage.py` | the harness, beside the notebook | holds the candidate prompts and schemas; reads its cost from the trace |

**Run it:**

```bash
cd labs/session-04/project
uv run python triage.py --limit 3
```


In [38]:
done = subprocess.run([sys.executable, "triage.py", "--limit", "3"],
                      cwd=HERE / "project", capture_output=True, text=True, timeout=900)
print(done.stdout[-4200:] or done.stderr[-1200:])

ShopWise Support Assistant v1.0.0 · google_genai:gemini-3.1-flash-lite · 3 ticket(s)
  tracing -> project 'shopwise-cce-ind-tra-2026'
  prompt  -> v1 · sha 41dcfc647975

TCK-001  order_status | medium | calm | needs_human=False
  Customer inquiring about status of order ORD-5001 due to lack of tracking updates.
--------------------------------------------------------------------------
Hello,

Thank you for reaching out, and I am sorry that your order has not arrived yet. I understand it is frustrating to see that your tracking information has not updated.

I do not have access to our courier tracking system, so I am passing your ticket to a human colleague who can look into this for you. They will review your order status and email you with an update, usually within one working day.

ShopWise Support

TCK-002  refund | medium | calm | needs_human=True
  Customer requests a refund for headphones with a crackling left earcup.
--------------------------------------------------------------

Six live calls, three tickets, and the vendor appears only as the string it was configured with.
Change two lines in `labs/.env` — `SHOPWISE_PROVIDER=groq`, `SHOPWISE_MODEL=openai/gpt-oss-20b` — and
the same command runs on a different company's hardware with the same schema, prompts, stamp and
traces. That was the sprint plan. It is now a text edit.

**And the limitation:** read `TCK-001`'s reply. Correctly triaged, polite, escalated — and it does not
answer the question. Hold that for section 12.4.


## 12. The wrap

### 12.1 The docs, and the four-step move

1. **Find the page** on <https://docs.langchain.com/oss/python/langchain/> — organised by concept
   (`models`, `messages`, `structured-output`, `observability`).
2. **Switch to the Python tab.**
3. **Take the smallest example** that shows the thing, not the one nearest your problem.
4. **Run it unchanged before adapting it.** Fails unchanged → your environment. Works unchanged, fails
   adapted → your adaptation.

Performed on streaming — the docs' example, step 4, unchanged:


In [39]:
# The docs' streaming example, unchanged: one loop, one chunk at a time.
for chunk in model.stream("Name three things a support assistant cannot do. One line each."):
    print(chunk.text, end="", flush=True)

A support assistant cannot perform physical tasks in the real world.
A

 support assistant cannot access or modify your private local files without explicit permission and integration.
A support assistant cannot experience human emotions, subjective feelings, or

 consciousness.

Step 5, which the docs cannot do for you: adapt it and notice what changes.


In [40]:
started = time.perf_counter()
whole = None

for i, chunk in enumerate(model.stream([("system", project.PROMPTS["system_prompt"]),
                                        ("human", as_block(TICKET))])):
    # Chunks ADD. Accumulating them gives you back an ordinary message at the end.
    whole = chunk if whole is None else whole + chunk
    print(f"  chunk {i:>2} at {time.perf_counter() - started:>5.2f}s   "
          f"{len(chunk.text):>4} chars")

print(f"\nwhole reply after {time.perf_counter() - started:.2f}s, {len(whole.text)} chars")
print(f"accumulated type  {type(whole).__name__}")
print(f"usage_metadata    {whole.usage_metadata}\n")
print(f"{whole.text[:200].strip()}...")

  chunk  0 at  0.90s      6 chars
  chunk  1 at  0.90s     77 chars
  chunk  2 at  0.90s    119 chars
  chunk  3 at  0.90s    123 chars
  chunk  4 at  0.98s    135 chars
  chunk  5 at  1.05s    116 chars
  chunk  6 at  1.05s      0 chars
  chunk  7 at  1.05s      0 chars

whole reply after 1.06s, 576 chars
accumulated type  AIMessageChunk
usage_metadata    {'input_tokens': 619, 'output_tokens': 121, 'total_tokens': 740, 'input_token_details': {'cache_read': 0}}

Hello,

I am sorry to hear that you have been waiting three weeks for your order. I understand how disappointing it is to be left waiting for an item that has not arrived.

I do not have access to ord...


**Streaming never makes a model faster.** What moves is *when the first words exist* — the gap
between first and last chunk is the interval a streaming UI shows a reply and a spinner UI shows a
spinner. It is worth more the longer the answer; on a two-sentence reply it is worth nothing. Chunks
**add** (`chunk + chunk`), and `usage_metadata` arrives on the last, text-less ones — concatenate
`.text` and you lose the bill. Session 6 does streaming properly.

### 12.2 The card: v1 versus the internet

| you will find online (pre-2025) | what we use (LangChain v1 / LangGraph 1.x) |
|---|---|
| `create_react_agent`, `AgentExecutor`, `initialize_agent` | `create_agent` (from `langchain.agents`) — session 5 |
| `LLMChain`, `ConversationChain`, `SequentialChain` | gone — `create_agent` or plain `model.invoke()` |
| `from langchain_core.messages import …` | `from langchain.messages import …` |
| `from langchain_core.tools import tool` | `from langchain.tools import tool` — session 5 |
| `ConversationBufferMemory`, `memory=` | a checkpointer + `thread_id`, `store=` for long-term — session 6 |
| `Annotated[…, InjectedState]`, `get_runtime()` | a `ToolRuntime` parameter — session 5 |
| `.astream_events(version="v1"/"v2")` | `stream_events(…, version="v3")` — session 6 |
| callbacks for cross-cutting behaviour | **middleware** — session 7 |
| `prompt=` | `system_prompt=` |

A card is a claim like any other — check it against what is installed:


In [41]:
import importlib
import re

# The right-hand column of the card, imported for real out of the pinned packages.
for module, name in [("langchain.agents", "create_agent"),
                     ("langchain.tools", "tool"),
                     ("langchain.tools", "ToolRuntime"),
                     ("langchain.messages", "SystemMessage"),
                     ("langchain.chat_models", "init_chat_model")]:
    present = hasattr(importlib.import_module(module), name)
    print(f"  {'ok ' if present else 'NO '} from {module} import {name}")

print(f"\nchecked against langchain {langchain.__version__} · "
      f"langchain-core {langchain_core.__version__}")

# And the line that keeps the card true — read out of the file rather than quoted from memory.
pins = re.findall(r'"(lang(?:chain|graph)[<>=0-9.,]+)"',
                  (HERE.parent / "pyproject.toml").read_text(encoding="utf-8"))
print(f"pinned in labs/pyproject.toml: {', '.join(pins)}")

  ok  from langchain.agents import create_agent
  ok  from langchain.tools import tool
  ok  from langchain.tools import ToolRuntime
  ok  from langchain.messages import SystemMessage
  ok  from langchain.chat_models import init_chat_model

checked against langchain 1.3.14 · langchain-core 1.6.1
pinned in labs/pyproject.toml: langchain>=1.3.14,<2, langgraph>=1.2.10,<1.3


### 12.3 Recap: v0.2 → v1

| | v0.2 — the raw SDK | v1 — LangChain |
|---|---|---|
| the vendor | in every file of the program | one string in `labs/.env` |
| a second vendor | rewrite both calls, the client, the cost report, the errors | change the string |
| structured output | one spelling per vendor | `with_structured_output`, five vendors |
| the schema file | `schemas.py` | byte-identical `schemas.py` |
| tokens and cost | `cost()` + a hand-maintained `prices.yml` | read from the trace; the eval reads the same records |
| debugging | `print`, and rerunning | the run list, including runs that already finished |
| which prompt produced this | nobody knew | `prompt_version` + `prompt_sha` on every run |
| a prompt change | edit, hope | score both arms over 20 tickets, then decide |
| errors | `ClientError` vs `ServerError` | one branch that prints the class name |
| accuracy on 20 tickets | 18/20 | 17–18/20 — the same band |

### 12.4 The hook: Ayesha's order

> **"Hi, this is Ayesha. Where is my order ORD-5001? When will it be delivered?"**


In [42]:
AYESHA = {
    "id": "TCK-HOOK",
    "subject": "Where is my order ORD-5001?",
    "body": "Hi, this is Ayesha. Where is my order ORD-5001? When will it be delivered?",
}

result = project.triage(model, AYESHA)
print(f"triage   {result.category} | {result.urgency} | {result.sentiment} | "
      f"needs_human={result.needs_human}\n")
print(project.draft_reply(model, AYESHA, result))

triage   order_status | low | calm | needs_human=False



Hello Ayesha,

Thank you for reaching out about your order. I understand you are looking for an update on when your package will arrive.

I do not have access to shipping information or order tracking. I am passing your ticket to a human colleague who can see these details for you.

They will review your order and reply to this email, usually within one working day.

ShopWise Support


**It did everything right and answered nothing.** Every order-status ticket becomes a human's job —
the job the assistant was built to do. `ORD-5001` is a real order in ShopWise's records; the assistant
has no way to reach it. It can read, reason and write, and it cannot *look anything up*.

Why did it not invent an answer? Because one paragraph of English told it not to — `prompts.yml`,
block 4. Take that paragraph away, same model, same question:


In [43]:
# The same model and the same question, with one sentence of role instead of the eight-block
# prompt: no guardrails, no output format, nothing telling it what it may not claim. Three runs,
# because one run of a generative model is an anecdote and this claim is worth more than that.
UNGUARDED = ("You are the customer support assistant for ShopWise, an online electronics "
             "store. Reply to the customer's email.")

for attempt in range(1, 4):
    naked = model.invoke([("system", UNGUARDED),
                          ("human", project.as_ticket_block(AYESHA))])
    print(f"{'=' * 74}\nRUN {attempt}\n{'=' * 74}")
    print(naked.text.strip())
    print()

RUN 1
Subject: Update regarding your ShopWise order ORD-5001

Hi Ayesha,

Thank you for reaching out to ShopWise!

I have looked into your order (ORD-5001) and I am happy to provide you with an update. Your package is currently in transit and is estimated to be delivered by [Insert Date].

You can track the live progress of your shipment using the link below:
[Insert Tracking Link]

If you have any further questions or if there is anything else I can assist you with, please feel free to reply to this email.

Best regards,

Customer Support Team
ShopWise



RUN 2
Subject: Update regarding your order ORD-5001

Hi Ayesha,

Thank you for reaching out to ShopWise!

I would be happy to look into the status of your order for you. I have checked our system, and I can confirm that your order (ORD-5001) is currently being processed at our distribution center.

It is estimated to be delivered by **[Insert Date]**. Once your order ships, you will receive a separate email containing your tracking number so you can follow its progress directly.

If you have any further questions in the meantime, please feel free to reply to this email. We appreciate your patience!

Best regards,

Customer Support Team
ShopWise



RUN 3
Subject: Re: Where is my order ORD-5001?

Hi Ayesha,

Thank you for reaching out to ShopWise!

I have looked into your order, **ORD-5001**. I am happy to inform you that your package is currently in transit. You can track the real-time progress of your delivery using the link below:

[Link to Tracking Portal/Carrier Website]

Your order is currently estimated to arrive by **[Insert Date]**. 

If you have any further questions or if your package does not arrive by the expected date, please feel free to reply to this email, and I will be happy to assist you further.

Best regards,

[Your Name]
Customer Support Team
ShopWise



**Every run claims access it does not have** — *"I have looked into your order"* — asserts a
status, and puts something in the delivery-date hole: a `[Insert Date]` placeholder, or on another run
a specific Thursday for a parcel nobody looked at. The failure is not the invented date. It is that
something had to go in that sentence and nothing true was available. **Next session: tools.**


---

# 🎓 Advanced appendix

Optional, self-paced, and none of it is needed for session 5. Six pieces: the one that makes you
better at *arguing* about frameworks comes first, then four capabilities the core session skipped,
then a build.

## A1. Where the five providers genuinely differ

The standard interface is real — you watched it be real. It is also **thinner than it looks**, and
the differences underneath are not bugs. Here is everything this notebook actually measured, in one
table, so the next time somebody says *"we can swap providers, it is one string"* you can agree and
then say what else moves.

| | Gemini | Groq | OpenAI | Anthropic | Ollama |
|---|---|---|---|---|---|
| structured output, **how** | native JSON schema, enforced at the decoder | **forced tool call** | native JSON mode, answer in the text | **forced tool call** | native JSON, answer in the text |
| the schema's price, input tokens | **+0** | +282 | +305 | **+947** | **+0** |
| where the answer arrives | `raw.text` | `raw.tool_calls[0]["args"]` | `raw.text` | `raw.tool_calls[0]["args"]` | `raw.text` |
| `usage_metadata` top level | identical | identical | identical | identical | identical |
| `input_token_details` | `cache_read` | — | `audio`, `cache_read` | 4 cache fields | — |
| `output_token_details` | — | `reasoning` | `audio`, `reasoning` | — | — |
| model name it answers as | as asked | as asked | **dated variant** you did not ask for | as asked | as asked |
| priced by the platform | yes | yes | yes | yes | **no — correctly** |

**Four things worth taking out of that table.**

**1. Mechanism and cost are different axes.** "Not a tool call" does not mean "free": OpenAI puts the
schema in the request as JSON-mode instructions and bills 305 tokens for it. The genuinely free case
is narrower — the provider that enforces the schema *at the decoder*, while tokens are being chosen,
so the schema never travels. Two of five do that.

**2. `raw.text` is empty on the tool-call providers.** If you had written your own portability layer,
this is the bug you would have shipped, and it would have looked like the model returning nothing.

**3. The dated model id is a billing problem, not a trivia question.** OpenAI answers as
`gpt-5.4-mini-<date>`, and that dated string is what a pricing table matches against. It lives in
`response_metadata`, the deliberately **non**-standardised dictionary — which tells you exactly how
much portability you were promised there.

**4. Reasoning tokens are invisible and fully billed.** Groq's `gpt-oss-20b` spent the large
majority of its output tokens on `reasoning` in section 5.2 — not in the answer, on the invoice, and
the reason that model returns an *empty reply* if `max_tokens` is set too low. You will meet this
again on every reasoning model.

### The error classes, which are the sharpest version of the point

The ledger's *got vaguer* row (section 10) said the error taxonomy lost precision, and that catching precisely means importing
the vendor coupling you paid to remove. That is checkable. Ask five providers for a model that does
not exist — a free request, since every one of them 4xx's before any tokens are generated — and read
the class names.

In [44]:
for provider_string, env_var, _note in PROVIDERS:
    if not available(env_var)[0]:
        print(f"{provider_string.split(':')[0]:<14} skipped, no key")
        continue
    vendor, _ = provider_string.split(":", 1)
    try:
        init_chat_model(f"{vendor}:does-not-exist", max_retries=0, timeout=20).invoke("hi")
        print(f"{vendor:<14} no error at all — which would be its own kind of problem")
    except Exception as err:
        cls = type(err)
        base = cls.__mro__[1]
        print(f"{vendor:<14} {cls.__module__}.{cls.__name__}")
        print(f"{'':<14}   which subclasses {base.__module__}.{base.__name__}")

google_genai   langchain_google_genai.chat_models.ChatGoogleGenerativeAIError
                 which subclasses langchain_google_genai._common.GoogleGenerativeAIError


groq           groq.NotFoundError
                 which subclasses groq.APIStatusError


openai         langchain_openai.chat_models.base.OpenAIModelNotFoundError
                 which subclasses openai.NotFoundError


anthropic      langchain_anthropic.chat_models.AnthropicModelNotFoundError
                 which subclasses anthropic.NotFoundError
ollama         ollama._types.ResponseError
                 which subclasses builtins.Exception


**Five providers, five different exception classes, from five different modules — and read the
second line of each pair, because that is where the argument is won.** Three of the five carry a
LangChain name, which looks like the abstraction covering for you. Two of those three subclass the
**vendor's own SDK exception** directly: `OpenAIModelNotFoundError` *is* an `openai.NotFoundError`.
The other two never pretended, and come straight out of `groq` and `ollama`.

There is no shared "model not found" type here to catch. `except Exception` and print the class name
is not laziness; it is the only vendor-neutral sentence available, and `triage.py`'s `main()` says
so in a comment rather than pretending otherwise.

The grown-up version of this — retry policies, fallbacks and error handling as *policy* rather than
`try`/`except` — arrives in session 7 as middleware. A6 below is the hand-rolled
version, so that the middleware means something when you meet it.

## A2. `content_blocks`, and the ticket with a photo

`.text` gives you the string. **`.content_blocks` gives you the typed pieces** — text, reasoning,
tool calls, citations, images — normalised into one shape across providers, which is the same trade
as `usage_metadata` one level up: a standard list of typed blocks, with a vendor-specific `extras`
escape hatch on each.

It matters for ShopWise sooner than you would think. Half of the real support tickets in an
electronics store arrive with a photo attached, and *"is this covered by the warranty"* is a
question about the picture, not about the text.

Sending one is a content block too.

In [45]:
import base64

photo = settings.DATA_DIR / "images" / "ticket-photo-headphones.jpg"

vision = model.invoke([{
    "role": "user",
    "content": [
        {"type": "text",
         "text": "A ShopWise customer attached this photo to a support ticket about their "
                 "headphones. In one sentence, what is damaged?"},
        {"type": "image",
         "base64": base64.b64encode(photo.read_bytes()).decode(),
         "mime_type": "image/jpeg"},
    ],
}])

print(vision.text)
print(f"\ncontent_blocks : {[b['type'] for b in vision.content_blocks]}")
print(f"usage          : {vision.usage_metadata['input_tokens']} in / "
      f"{vision.usage_metadata['output_tokens']} out")

The fabric covering on the ear cushion of the headphones is torn, exposing the foam padding underneath.

content_blocks : ['text']
usage          : 1104 in / 19 out


**The picture cost about a thousand input tokens** — more than the eight-block system prompt and a
whole ticket together, which the streamed call in section 12.1 billed at 619. An image is not a free
attachment, and at scale *"let customers attach photos"* is a budget decision before it is a feature
decision. Measure one on your provider before you promise it.

Two API notes:

- the same block list carries **PDFs** (`{"type": "file", ...}`) and audio on the providers that
  support them, and the integration raises rather than silently dropping when one does not;
- on the way *out*, `content_blocks` is where reasoning summaries and citations show up as their own
  typed blocks — which is how session 8's RAG work will read a citation without regexing prose.

Docs: **langchain/messages**, the content-block reference.

## A3. Throughput: batching, rate limits, and one model that is several

Three small things you will need the first time you run a job over more than ten tickets.

**`.batch()` runs them concurrently and returns them in order.** It is not a vendor batch API — it is
a thread pool around the same call — but the wall-clock difference on twenty tickets is the
difference between a coffee and a meeting.

In [46]:
tickets = project.load_tickets()[:3]
messages = [[("system", SYSTEM), ("human", as_block(t))] for t in tickets]
chain = model.with_structured_output(TriageResult)

started = time.perf_counter()
answers = chain.batch(messages)
batched = time.perf_counter() - started

started = time.perf_counter()
for one in messages:
    chain.invoke(one)
serial = time.perf_counter() - started

print(f"  batch()     {batched:>5.2f}s for 3 tickets")
print(f"  one by one  {serial:>5.2f}s")
print(f"  ->          {serial / batched:.1f}x faster\n")

# The order is the argument: results line up with the inputs, not with who finished first.
for ticket, answer in zip(tickets, answers):
    print(f"  {ticket['id']}  {answer.category:<14} {answer.urgency}")

  batch()      1.61s for 3 tickets
  one by one   3.72s
  ->          2.3x faster

  TCK-001  order_status   medium
  TCK-002  refund         high
  TCK-003  order_status   high


`batch_as_completed()` is the same thing yielding `(index, result)` **as each finishes**, which is
what you want behind a progress bar or when you are writing rows to a database rather than waiting
for the last one.

**And the thing that breaks batching: rate limits.** Groq's free tier is per-minute; Gemini's is too.
Twenty concurrent calls is how you find out. `InMemoryRateLimiter` is a token bucket you hand to the
model at construction — it blocks *before* the request, so nothing 429s in the first place. It needs
no model calls to demonstrate, which is the nicest thing about it.

In [47]:
from langchain_core.rate_limiters import InMemoryRateLimiter

# 2 requests per second, a bucket that holds 1. Nothing is called; we just take tokens.
limiter = InMemoryRateLimiter(requests_per_second=2, check_every_n_seconds=0.05, max_bucket_size=1)

started = time.perf_counter()
for i in range(4):
    limiter.acquire()                      # blocks until a token is available
    print(f"  request {i} allowed at {time.perf_counter() - started:.2f}s")

print("\n  -> half a second apart, and the first one waited too: the bucket starts empty.")
print("     Hand it in as init_chat_model(..., rate_limiter=limiter) and .batch() self-throttles.")

  request 0 allowed at 0.52s


  request 1 allowed at 1.03s


  request 2 allowed at 1.53s


  request 3 allowed at 2.04s

  -> half a second apart, and the first one waited too: the bucket starts empty.
     Hand it in as init_chat_model(..., rate_limiter=limiter) and .batch() self-throttles.


Two limitations to know before you rely on it: it is **in memory**, so it limits *this process* and
not your four API workers; and it counts **requests**, not tokens, so it cannot protect a
tokens-per-minute quota. Both are fine for a batch job and not fine for a fleet.

**Configurable models** are the third piece — one object whose model is chosen per call, rather than
per construction.

In [48]:
anymodel = init_chat_model(configurable_fields=("model", "model_provider"))

reply = anymodel.invoke(
    "Reply with exactly: ok",
    config={"configurable": {"model": settings.MODEL, "model_provider": settings.PROVIDER}},
)
print(f"{settings.PROVIDER}:{settings.MODEL} -> {reply.text.strip()!r}")
print("\nThe model was not chosen when the object was built. It was chosen at the call,")
print("from a dict — which is what you want when the choice comes from a request, a user")
print("tier, or an A/B assignment rather than from your settings file.")

google_genai:gemini-3.1-flash-lite -> 'ok'

The model was not chosen when the object was built. It was chosen at the call,
from a dict — which is what you want when the choice comes from a request, a user
tier, or an A/B assignment rather than from your settings file.


Where each belongs: `.batch()` for a job, `InMemoryRateLimiter` for a free tier, configurable models
for *"premium customers get the strong model"*. ShopWise's `settings.MODEL_STRONG` exists for exactly
that and nothing has used it yet.

Docs: **langchain/models** — the advanced section.

## A4. Tracing extras: scoping, and per-call labels

Section 9 stamped every run from inside `triage.py`. Two more controls are worth knowing, and both
live at the *call*, not in the file.

**Per-invoke tags and metadata** are the same `config=` the stamp uses. Anything you put there is
queryable afterwards — the customer's tier, the experiment arm, the deploy SHA. **`tracing_context`**
goes the other way: it switches recording off (or on) for a block, which is how you keep a load test,
a fixture-generation script or a run over real customer data out of the project.

```python
import langsmith as ls

# 1. one ordinary call, labelled at the call site
model.invoke("Say ok.", config={"tags": ["appendix", "a4-demo"],
                                "metadata": {"customer_tier": "gold", "arm": "control"}})

# 2. one call that must not be recorded
with ls.tracing_context(enabled=False):
    model.invoke("Say ok. This one is invisible.")
```

Two calls go out; the project shows one run, and it carries what you put on it:

```
tags          ['appendix', 'a4-demo']
my metadata   {'customer_tier': 'gold', 'arm': 'control'}
revision_id   'c790b0c-dirty'   <- the git commit this ran on
32 keys in total, most of them the tracer's own
```

**One run, not two.** The second call was made, answered and billed; it simply left no record,
because `tracing_context(enabled=False)` turned the tracer off for that block. That is the switch you
want around anything that would otherwise flood the project — and, more seriously, around anything
carrying data you are not allowed to store. *"We traced everything"* is a compliance statement as
much as an engineering one.

The metadata line is the other half. `customer_tier` and `arm` are not LangChain's fields; they are
yours, and they are queryable next week with the same `filter=` expression `score_triage.py` uses on its tags.
This is how a real incident gets narrowed: *the gold-tier customers on arm B, since Tuesday.*

**And notice `revision_id`, which nobody set.** The tracer read the git commit of the working tree
and recorded it on the run — `-dirty` and all. That is the join between *"this answer was bad"* and
*"this is the code that produced it"*, arriving for free, and it is worth knowing it exists before
the incident in which you need it.

Docs: **langchain/observability** — tracing context, tags and metadata.

## A5. Why v1 exists: the sixty-second history, at full length

Section 3.2 gave you the short version. Here is the whole shape, because it explains why the
internet's LangChain and this notebook's LangChain look like different libraries.

| | what the library was | what it looked like |
|---|---|---|
| **2022–2023** | a bag of *chains* | `LLMChain`, `ConversationChain`, `SequentialChain`, `RetrievalQA` — one class per pattern, and a new class every time somebody found a new pattern |
| **2023–2024** | chains **and** agents | `initialize_agent`, `AgentExecutor`, `create_react_agent` — plus LCEL and the `|` operator, an elegant idea that also became a second way to express everything |
| **2024–2025** | the split | control flow moved out to **LangGraph**, which is a graph engine, not a chain library; LangChain kept the components |
| **v1, October 2025** | one agent abstraction | the chain zoo retired to `langchain-classic`; `create_agent` is the agent; the public import surface moved to `langchain.*`; **middleware** replaces callbacks for cross-cutting behaviour |

**The engineering judgement worth stealing** is in the third row. Chains were a good idea that
generalised badly: every new control-flow shape needed a new class, and a class per pattern is a
library that grows without limit. The fix was not a better chain — it was to notice that *control
flow* and *components* are different problems, and to stop solving them in one library. LangGraph is
what "control flow" looks like when it is taken seriously (session 10), and `create_agent` is what
was left once it moved out (session 5).

**And the cost of that judgement is the reason you need the card in section 12.2.** Every tutorial
written before October 2025 teaches a library that no longer exists, from a package that still
installs, using imports that still work. A framework that is willing to break itself to get simpler
is the kind you want — and it hands you a migration problem roughly once a year. Pin the version,
read the release notes at upgrade time, and keep a card.

Docs: **langchain/philosophy** and **langchain/component-architecture**.

## A6. [ADV] Build: the fallback chain

**The problem, and it is a real one:** your primary provider has an incident. Not a slow response —
a genuine outage, an hour long, in the middle of a working day. Retries do not help, because the
thing you are retrying is down. Everything you built tonight makes the fix cheap, and this is the
exercise that proves it.

**Build it:**

1. Take a **deliberately broken** primary — a model id that does not exist is the honest test, since
   it fails exactly as an outage does and costs nothing.
2. Fall back to a working provider **from a different company**. A fallback to a second model at the
   same vendor protects you from a bad deploy; it does not protect you from that vendor being down.
3. **Measure the added latency**, because that is the number a reviewer will ask for. A fallback that
   costs four seconds on every call is not a resilience feature, it is a performance bug.
4. Then answer the question the code cannot: **which failures should fall back at all?** A 429 should
   probably wait rather than escape to a more expensive vendor. A validation error will fail
   identically on both. A timeout is the interesting one.

`with_fallbacks` is the one-line version. The cell below falls back **to the same vendor**, because
that is the one provider every reader of this notebook has — so it demonstrates the mechanism and
deliberately fails step 2. Crossing companies is the first item in *take it further*, and it is the
version worth having.

In [50]:
# A model id that does not exist: fails on the first request, every time, for free.
primary = init_chat_model("google_genai:gemini-does-not-exist", max_retries=0, timeout=20)
backup = init_chat_model(settings.model_id, timeout=settings.REQUEST_TIMEOUT_S,
                         max_retries=settings.MAX_RETRIES)

resilient = primary.with_fallbacks([backup])

# The overhead of a fallback is exactly how long the primary takes to fail. Time that on its own,
# rather than subtracting two whole calls from each other — two generations of different text
# differ by more than the thing we are trying to measure.
started = time.perf_counter()
try:
    primary.invoke("hi")
except Exception as err:
    refused = time.perf_counter() - started
    print(f"the primary failed after   {refused:>5.2f}s   ({type(err).__name__})")

started = time.perf_counter()
out = resilient.invoke([("system", SYSTEM), ("human", as_block(TICKET))])
end_to_end = time.perf_counter() - started

print(f"the chain answered after   {end_to_end:>5.2f}s   (failure + a real call to the backup)")
print(f"\n  {out.text[:100].strip()}...")

the primary failed after    0.75s   (ChatGoogleGenerativeAIError)


the chain answered after    3.12s   (failure + a real call to the backup)

  ### Ticket Triage Report: ORD-5001

**Priority:** High
**Category:** Order Status / Refund Request
*...


**The added latency is the whole finding, and the number above is the best case.** A refused model
id comes back fast, because the vendor rejects it before generating anything — so what you measured
is the *floor*. A real outage more often looks like a **timeout**, and then the fallback costs the
full `REQUEST_TIMEOUT_S` before the second call even starts, on every single request, until somebody
notices. Sixty seconds is the default in `config.py`.

That is why a fallback chain belongs behind a timeout you chose deliberately, and why *"just add a
fallback"* is not a free answer. It is also why the measurement above is written as two separate
timings rather than as one subtraction: two calls that generate different text differ by more than
the effect you are trying to see.

**Take it further, in your own repo:**

- fall back **across companies** (`google_genai` → `groq`), and confirm the schema still validates on
  the other side — section 5.3 says the token bill will not be the same;
- put the two arms behind a tag each and compare them in the trace, exactly as `score_triage.py`
  compares the A/B arms;
- decide what a fallback should do to `prompt_sha`. It does not change — same wording — which means
  the trace tells you the prompt was identical and the *provider* was not. That is the join you will
  want during the incident review.

**Where this returns:** session 7 replaces this with **`ModelFallbackMiddleware`**, which does the
same job as *policy* attached to an agent rather than as a wrapper around a model — along with retry,
summarisation and human-in-the-loop, all as middleware. Having hand-rolled the fallback once is what
makes that middleware read as a convenience rather than as magic.